# Hallucination: AI jhooth kyun bolta hai?
**AI Explained · Episode 3 (full version)** · Jajati AI

Har cell ko upar se neeche chalao (Shift + Enter). Google Colab mein free chalta hai, kuch install nahi karna.

Hum ek **chhota sa AI** banayenge: ye sirf ginta hai ki kaunsa word kiske baad aata hai. Asli AI (LLM) isse karodon guna bada hai, par idea wahi hai: agla word wo chuno jo sabse *sahi lage*.

In [1]:
facts = [
    "Bharat ki rajdhani Delhi hai .",
    "Bharat ki rajdhani Delhi hai .",
    "Bharat ki rajdhani Delhi hai .",
    "France ki rajdhani Paris hai .",
    "Japan ki rajdhani Tokyo hai .",
    "Nepal ki rajdhani Kathmandu hai .",
]
print(len(facts), "sentences, sab sach")

6 sentences, sab sach


In [2]:
from collections import Counter, defaultdict

def train(facts):
    model = defaultdict(Counter)
    for s in facts:
        w = s.split()
        for n in (1, 2, 3):
            for i in range(n, len(w)):
                ctx = tuple(w[i - n:i])
                model[ctx][w[i]] += 1
    return model

model = train(facts)
print(len(model), "patterns yaad kiye")

36 patterns yaad kiye


In [3]:
def probs(text, model=model):
    w = text.split()
    for n in (3, 2, 1):
        ctx = tuple(w[-n:])
        if ctx in model:
            c = model[ctx]
            total = sum(c.values())
            return {k: round(100 * v / total)
                    for k, v in c.most_common()}

print(probs("Japan ki rajdhani"))

{'Tokyo': 100}


In [4]:
print(probs("Australia ki rajdhani"))

{'Delhi': 50, 'Paris': 17, 'Tokyo': 17, 'Kathmandu': 17}


In [5]:
def complete(text, model=model):
    w = text.split()
    while w[-1] != ".":
        p = probs(" ".join(w), model)
        w.append(max(p, key=p.get))
    return " ".join(w)

print(complete("Japan ki rajdhani"))
print(complete("Australia ki rajdhani"))

Japan ki rajdhani Tokyo hai .
Australia ki rajdhani Delhi hai .


In [6]:
words = {w for s in facts for w in s.split()}
print("'pata' seekha?", "pata" in words)

'pata' seekha? False


In [7]:
import random
random.seed(3)

def sample(text):
    p = probs(text)
    return random.choices(list(p),
                          weights=p.values())[0]

for _ in range(5):
    print("Australia ki rajdhani",
          sample("Australia ki rajdhani"))

Australia ki rajdhani Delhi
Australia ki rajdhani Paris
Australia ki rajdhani Delhi
Australia ki rajdhani Paris
Australia ki rajdhani Paris


In [8]:
random.seed(1)
# 100 sawal, jawab nahi pata, 4 options
guess = sum(random.random() < 0.25
            for _ in range(100))
print("Guess kiya:", guess, "marks")
print("'Pata nahi' bola:", 0, "marks")

Guess kiya: 24 marks
'Pata nahi' bola: 0 marks


## Fix: AI ko document do
Model ke paas sahi baat ho, to wo pattern se guess nahi karta. Asli AI mein ye kaam RAG karta hai (Episode 4): sawal ke saath sahi document bhi desk par rakh do.

In [9]:
facts.append("Australia ki rajdhani Canberra hai .")
model = train(facts)
print(complete("Australia ki rajdhani", model))

Australia ki rajdhani Canberra hai .


In [10]:
# Apna sawal daalo (jaise "Italy ki rajdhani"):
q = "Italy ki rajdhani"
print(probs(q, model))
print(complete(q, model))

{'Delhi': 43, 'Paris': 14, 'Tokyo': 14, 'Kathmandu': 14, 'Canberra': 14}
Italy ki rajdhani Delhi hai .
